In [1]:
%pip install pandas pymupdf sentence-transformers faiss-cpu


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# datset check
from pathlib import Path
import os

DATA_PATH = Path("arxiv-metadata-oai-snapshot.json")

print("Current folder:", os.getcwd())
print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("Dataset size (GB):",
          round(DATA_PATH.stat().st_size / (1024**3), 2))
else:
    print("Dataset current folder mein nahi mila.")

Current folder: C:\Users\lucky\Downloads\Research Paper Q&A
Dataset exists: True
Dataset size (GB): 3.53


In [3]:
import json
import pandas as pd

papers = []

with open("arxiv-metadata-oai-snapshot.json", "r", encoding="utf-8") as file:
    for line in file:
        try:
            item = json.loads(line)
            categories = item.get("categories", "").split()

            if any(
                category.startswith("cs.")
                or category == "stat.ML"
                for category in categories
            ):
                papers.append({
                    "id": item.get("id"),
                    "title": item.get("title"),
                    "abstract": item.get("abstract"),
                    "categories": item.get("categories")
                })

            if len(papers) >= 100:
                break

        except (json.JSONDecodeError, AttributeError):
            continue

cs_papers = pd.DataFrame(papers)

print("Relevant CS/ML papers:", len(cs_papers))
display(cs_papers[["id", "title", "categories"]].head(10))

Relevant CS/ML papers: 100


,id,title,categories
0,0704.0002,Sparsity-certifying Graph Decompositions,math.CO cs.CG
1,0704.0046,A limit relation for entropy and channel capac...,quant-ph cs.IT math.IT
2,0704.0047,Intelligent location of simultaneously active ...,cs.NE cs.AI
3,0704.0050,Intelligent location of simultaneously active ...,cs.NE cs.AI
4,0704.0062,On-line Viterbi Algorithm and Its Relationship...,cs.DS
5,0704.0090,Real Options for Project Schedules (ROPS),cs.CE cond-mat.stat-mech cs.MS cs.NA physics.d...
6,0704.0098,Sparsely-spread CDMA - a statistical mechanics...,cs.IT math.IT
7,0704.0108,Reducing SAT to 2-SAT,cs.CC
8,0704.0213,Geometric Complexity Theory V: On deciding non...,cs.CC
9,0704.0217,Capacity of a Multiple-Antenna Fading Channel ...,cs.IT math.IT


In [4]:
%pip install -U openai

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------ --------- 1.8/2.4 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.5 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 3.37.0
    Uninstalling openai-3.37.0:
      Successfully uninstalled openai-3.37.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# connection set karo

import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "API key nahi mili. Jupyter restart check karo."
    )

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully!")

OpenAI client initialized successfully!


In [6]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Reply with: ResearchLens AI local model connected!",
        "stream": False
    },
    timeout=120
)

response.raise_for_status()
print(response.json()["response"])

ResearchLens AI local model connected!


In [7]:
# FAISS + Llama ko connect karo

def ask_researchlens(question, top_k=3):
    # Step 1: Retrieve relevant passages using existing FAISS
    results = search_paper(question, top_k=top_k)

    if not results:
        return "Paper mein relevant information nahi mili."

    # Step 2: Build context with page references
    context = "\n\n".join(
        f"[Page {result['page']}]\n{result['text']}"
        for result in results
    )

    # Step 3: Ask local Llama to answer from retrieved context
    prompt = f"""
You are ResearchLens AI, a research paper assistant.

Answer the question using ONLY the context below.
If the context does not contain the answer, say that
the information was not found in the retrieved passages.
Include the relevant page number in your answer.
Do not invent facts.

Context:
{context}

Question: {question}

Answer:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        },
        timeout=180
    )

    response.raise_for_status()
    answer = response.json()["response"]

    return answer

In [8]:
print("search_paper" in globals())
print("chunks" in globals())
print("index" in globals())

False
False
False


In [9]:
print("search_paper:", "search_paper" in globals())
print("chunks:", "chunks" in globals())
print("index:", "index" in globals())

search_paper: False
chunks: False
index: False


In [10]:
import os
import fitz

pdf_path = r"C:\Users\lucky\Downloads\Research Paper Q&A\0704.2083.pdf"

print("PDF exists:", os.path.exists(pdf_path))

PDF exists: True


In [11]:
# pdf ko dobara extracct karo

import pymupdf

doc = pymupdf.open(pdf_path)

pages = []

for page_num, page in enumerate(doc):
    text = page.get_text()
    pages.append({
        "page": page_num + 1,
        "text": text
    })

doc.close()

print("Total pages:", len(pages))
print("First page text preview:")
print(pages[0]["text"][:500])

Total pages: 4
First page text preview:
  
Abstract— In this paper Arabic was investigated from the 
speech recognition problem point of view.  We propose a 
novel approach to build an Arabic Automated Speech 
Recognition System (ASR). This system is based on the open 
source CMU Sphinx-4, from the Carnegie Mellon University. 
CMU Sphinx is a large-vocabulary; speaker-independent, 
continuous speech recognition system based on discrete 
Hidden Markov Models (HMMs). We build a model using 
utilities from the OpenSource CMU Sphinx. We w


In [12]:
# Text ko chunks mein divide karo

def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = []

for page in pages:
    page_chunks = chunk_text(
        page["text"],
        chunk_size=1000,
        overlap=200
    )

    for chunk_id, text in enumerate(page_chunks):
        chunks.append({
            "page": page["page"],
            "chunk_id": chunk_id,
            "text": text
        })

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0]["text"][:300])

Total chunks: 21

First chunk preview:

Abstract— In this paper Arabic was investigated from the 
speech recognition problem point of view.  We propose a 
novel approach to build an Arabic Automated Speech 
Recognition System (ASR). This system is based on the open 
source CMU Sphinx-4, from the Carnegie Mellon University. 
CMU Sphinx is 


In [13]:
# Embeddings dobara banao

import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (21, 384)


In [14]:
# FAISS index recreate karo
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors:", index.ntotal)
print("Vector dimensions:", index.d)

FAISS index created successfully!
Total vectors: 21
Vector dimensions: 384


In [15]:
# search_paper() function restore karo
def search_paper(question, top_k=3):
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, index.ntotal)
    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        results.append({
            "score": float(score),
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"]
        })

    return results


print("search_paper() restored successfully!")

search_paper() restored successfully!


In [16]:
# FAISS search test karo
results = search_paper(
    "Which speech recognition system is used in this paper?",
    top_k=3
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i} | Page {result['page']} | Score: {result['score']:.3f}")
    print(result["text"][:400])
    print("-" * 60)


Result 1 | Page 1 | Score: 0.634
s that a person speaks 
into a microphone or telephone.  It has a wide area of 
applications: Command recognition (Voice user interface with 
the computer), Dictation, Interactive Voice Response, it can 
be used to learn a foreign language. ASR can help also, 
handicapped people to interact with society. It is a technology 
which makes life easier and very promising [1]. 
View the importance of AS
------------------------------------------------------------

Result 2 | Page 4 | Score: 0.632
eddy, “An overview of the SPHINX 
speech recognition system,” IEEE Transactions on Acoustics, Speech 
and Signal Processing, vol. 38, no. 1, pp. 35–45, Jan. 1990. 
[6] 
X. Huang, F. Alleva, H. W. Hon, M. Y. Hwang, and R. Rosenfeld, “The 
SPHINX-II speech recognition system: an overview,” Computer Speech 
and Language, vol. 7, no. 2, pp. 137–148, 1993. 
[7] 
M. K. Ravishankar, “Efficient algorithms
------------------------------------------------------------

Result 

In [17]:
# FAISS ke results ko inspect karo

results = search_paper(
    "CMU Sphinx-4 Arabic Automated Speech Recognition System",
    top_k=5
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i} | Page {result['page']} | Score: {result['score']:.3f}")
    print(result["text"][:250])
    print("-" * 50)


Result 1 | Page 1 | Score: 0.881
Abstract— In this paper Arabic was investigated from the 
speech recognition problem point of view.  We propose a 
novel approach to build an Arabic Automated Speech 
Recognition System (ASR). This system is based on the open 
source CMU Sphinx-4, fr
--------------------------------------------------

Result 2 | Page 3 | Score: 0.865
nd there are as many syllables in a word as 
vowels in it [23]. 
B. Hello Arabic Digit application 
 
 
 
Table 1:  Ten Arabic digits, how to pronounce them, type of 
syllable, IPA representation [24] and number of syllables in 
every spoken digit [2
--------------------------------------------------

Result 3 | Page 1 | Score: 0.844
gets to quickly begin 
conducting research and developing applications. In this work 
we attempt to build an ASR based on CMU Sphinx for the 
Arabic language. We describe our experience for the 
extension of this system into Arabic language. 
 
II. P
-------------------------------------------

In [18]:
# ResearchLens AI ka final RAG function banao
import requests

def ask_researchlens(question, top_k=3):
    # 1. Retrieve relevant passages from FAISS
    results = search_paper(question, top_k=top_k)

    if not results:
        return "Relevant information paper mein nahi mili."

    # 2. Prepare context with page citations
    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    # 3. Generate an answer using local Llama
    prompt = f"""
You are ResearchLens AI, an assistant for research papers.


Answer the question using only the context below.
If the answer is not present, say that it was not found.
Cite sources ONLY using the exact page labels provided in the context,
such as [Page 1]. Never use bibliography reference numbers like [5-7].
Do not invent facts.

Context:
{context}

Question: {question}

Answer:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        },
        timeout=180
    )

    response.raise_for_status()
    return response.json()["response"]


print("ResearchLens AI RAG function ready!")

ResearchLens AI RAG function ready!


In [19]:
# Llama ko direct context test do
test_context = """
The proposed Arabic Automated Speech Recognition System
is based on the open-source CMU Sphinx-4 system
from Carnegie Mellon University.
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": (
            "Answer using only the given context. "
            "Which speech recognition system is used? "
            "Give a short, direct answer.\n\n"
            f"Context:\n{test_context}\n\n"
            "Answer:"
        ),
        "stream": False
    },
    timeout=180
)

response.raise_for_status()
print(response.json()["response"])

CMU Sphinx-4.


In [20]:
# ask_researchlens() function update karo

import requests

def ask_researchlens(question, top_k=3):
    results = search_paper(question, top_k=top_k)

    if not results:
        return "Relevant information paper mein nahi mili."

    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    prompt = f"""You are ResearchLens AI.
Answer the question using only the provided context.
Give a concise answer. Do not include citations or reference numbers.
If the answer is not in the context, say it was not found.

Context:
{context}

Question: {question}

Answer:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 120
            }
        },
        timeout=180
    )

    response.raise_for_status()
    answer = response.json()["response"].strip()

    # Add actual PDF page citations from retrieved passages
    page_numbers = sorted(set(r["page"] for r in results))
    citations = ", ".join(f"[Page {p}]" for p in page_numbers)

    return f"{answer}\n\nSource: {citations}"


print("Citation-enabled ResearchLens AI ready!")


Citation-enabled ResearchLens AI ready!


In [21]:
print(ask_researchlens(
    "Which speech recognition system is used in this paper?"
))

The speech recognition system used in this paper is CMU Sphinx-4, which is based on Hidden Markov Models (HMMs).

Source: [Page 1], [Page 4]


In [22]:
# ui banana hain
%pip install gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
# Gradio interface ka code
import gradio as gr

def chat_with_researchlens(question):
    if not question or not question.strip():
        return "Please enter a question."

    try:
        answer = ask_researchlens(question.strip())
        return answer
    except Exception as e:
        return f"Error: {str(e)}"


demo = gr.Interface(
    fn=chat_with_researchlens,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a question about your research paper...",
        label="Your Question"
    ),
    outputs=gr.Textbox(
        lines=6,
        label="ResearchLens AI Answer"
    ),
    title="🔬 ResearchLens AI",
    description="Ask questions about a research paper using FAISS retrieval and a local Llama 3.2 model.",
    examples=[
        "Which speech recognition system is used in this paper?",
        "What are Hidden Markov Models used for?",
        "What is the main objective of this paper?"
    ]
)

print("Gradio interface created successfully!")

Gradio interface created successfully!


In [24]:
# ResearchLens AI launch karo
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [25]:
# PDF upload feature ke liye imports check karo
import gradio as gr
import pymupdf
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("All PDF processing libraries are ready!")

All PDF processing libraries are ready!


In [26]:
# PDF processing function banao

def process_uploaded_pdf(pdf_path):
    # 1. Extract text from the uploaded PDF
    doc = pymupdf.open(pdf_path)
    uploaded_pages = []

    for page_num, page in enumerate(doc):
        text = page.get_text().strip()

        if text:
            uploaded_pages.append({
                "page": page_num + 1,
                "text": text
            })

    doc.close()

    if not uploaded_pages:
        raise ValueError(
            "PDF se text extract nahi hua. "
            "Ho sakta hai PDF scanned image ho."
        )

    # 2. Split text into overlapping chunks
    uploaded_chunks = []

    for page in uploaded_pages:
        page_chunks = chunk_text(
            page["text"],
            chunk_size=1000,
            overlap=200
        )

        for chunk_id, text in enumerate(page_chunks):
            uploaded_chunks.append({
                "page": page["page"],
                "chunk_id": chunk_id,
                "text": text
            })

    # 3. Create embeddings using the existing model
    chunk_texts = [c["text"] for c in uploaded_chunks]

    uploaded_embeddings = model.encode(
        chunk_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # 4. Build a separate FAISS index
    uploaded_index = faiss.IndexFlatIP(
        uploaded_embeddings.shape[1]
    )
    uploaded_index.add(uploaded_embeddings)

    return uploaded_chunks, uploaded_index


print("PDF processing function ready!")


PDF processing function ready!


In [27]:
# Uploaded PDF ka search function banao

def search_uploaded_pdf(question, uploaded_chunks, uploaded_index, top_k=3):
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, uploaded_index.ntotal)
    scores, indices = uploaded_index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        results.append({
            "score": float(score),
            "page": uploaded_chunks[idx]["page"],
            "text": uploaded_chunks[idx]["text"]
        })

    return results


print("Uploaded PDF search function ready!")


Uploaded PDF search function ready!


In [28]:
# Uploaded PDF ke liye answer function banao

def ask_uploaded_pdf(question, uploaded_chunks, uploaded_index, top_k=3):
    results = search_uploaded_pdf(
        question,
        uploaded_chunks,
        uploaded_index,
        top_k=top_k
    )

    if not results:
        return "Uploaded PDF mein relevant information nahi mili."

    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    prompt = f"""You are ResearchLens AI, a research paper assistant.

Answer the question using only the context below.
If the answer is not present, say it was not found.
Do not include bibliography reference numbers.
Give a concise answer.

Context:
{context}

Question: {question}

Answer:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 120
            }
        },
        timeout=180
    )

    response.raise_for_status()
    answer = response.json()["response"].strip()

    # Add page citations from the retrieved PDF passages
    page_numbers = sorted(set(r["page"] for r in results))
    citations = ", ".join(f"[Page {p}]" for p in page_numbers)

    return f"{answer}\n\nSource: {citations}"


print("Uploaded PDF answer function ready!")


Uploaded PDF answer function ready!


In [29]:
# Gradio interface ko PDF upload ke saath update karo

import gradio as gr

def answer_from_upload(pdf_file, question):
    if pdf_file is None:
        return "Please upload a PDF first."

    if not question or not question.strip():
        return "Please enter a question."

    try:
        uploaded_chunks, uploaded_index = process_uploaded_pdf(pdf_file)

        return ask_uploaded_pdf(
            question.strip(),
            uploaded_chunks,
            uploaded_index
        )
    except Exception as e:
        return f"Error: {str(e)}"


upload_demo = gr.Interface(
    fn=answer_from_upload,
    inputs=[
        gr.File(
            label="Upload Research Paper (PDF)",
            file_types=[".pdf"],
            type="filepath"
        ),
        gr.Textbox(
            lines=2,
            label="Your Question",
            placeholder="Ask a question about the uploaded paper..."
        )
    ],
    outputs=gr.Textbox(
        lines=6,
        label="ResearchLens AI Answer"
    ),
    title="ResearchLens AI — PDF Q&A",
    description="Upload a research paper and ask questions using FAISS and local Llama 3.2."
)

print("PDF upload interface created successfully!")


PDF upload interface created successfully!


In [30]:
# PDF upload interface launch karo
upload_demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [31]:
# PDF processing ko cache karne ke liye function banao
from functools import lru_cache
import os
import hashlib

@lru_cache(maxsize=4)
def get_processed_pdf(pdf_path, file_signature):
    uploaded_chunks, uploaded_index = process_uploaded_pdf(pdf_path)
    return uploaded_chunks, uploaded_index


def load_pdf_cached(pdf_path):
    file_size = os.path.getsize(pdf_path)

    with open(pdf_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()

    return get_processed_pdf(
        pdf_path,
        (file_size, file_hash)
    )


print("PDF caching functions ready!")

PDF caching functions ready!


In [32]:
# answer_from_upload() function update karo
def answer_from_upload(pdf_file, question):
    if pdf_file is None:
        return "Please upload a PDF first."

    if not question or not question.strip():
        return "Please enter a question."

    try:
        # Reuse cached chunks and FAISS index
        uploaded_chunks, uploaded_index = load_pdf_cached(pdf_file)

        return ask_uploaded_pdf(
            question.strip(),
            uploaded_chunks,
            uploaded_index
        )

    except Exception as e:
        return f"Error: {str(e)}"


print("Cached PDF answer function ready!")

Cached PDF answer function ready!


In [33]:
# Updated interface restart karo
upload_demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


In [34]:
# Retrieved passages inspect karo
test_chunks, test_index = load_pdf_cached(pdf_path)

results = search_uploaded_pdf(
    "Which speech recognition system is used in this paper?",
    test_chunks,
    test_index,
    top_k=3
)

for r in results:
    print(f"\nPage: {r['page']} | Score: {r['score']:.3f}")
    print(r["text"][:500])
    print("-" * 60)


Page: 4 | Score: 0.641
, “An overview of the SPHINX 
speech recognition system,” IEEE Transactions on Acoustics, Speech 
and Signal Processing, vol. 38, no. 1, pp. 35–45, Jan. 1990. 
[6] 
X. Huang, F. Alleva, H. W. Hon, M. Y. Hwang, and R. Rosenfeld, “The 
SPHINX-II speech recognition system: an overview,” Computer Speech 
and Language, vol. 7, no. 2, pp. 137–148, 1993. 
[7] 
M. K. Ravishankar, “Efficient algorithms for speech recognition,” PhD 
Thesis (CMU Technical Report CS-96-143), Carnegie Mellon 
University, Pit
------------------------------------------------------------

Page: 1 | Score: 0.627
hat a person speaks 
into a microphone or telephone.  It has a wide area of 
applications: Command recognition (Voice user interface with 
the computer), Dictation, Interactive Voice Response, it can 
be used to learn a foreign language. ASR can help also, 
handicapped people to interact with society. It is a technology 
which makes life easier and very promising [1]. 
View the importanc

In [35]:
# Search results ko inspect aur filter karo
results = search_uploaded_pdf(
    "CMU Sphinx-4 system used for Arabic speech recognition",
    test_chunks,
    test_index,
    top_k=5
)

for r in results:
    print(f"\nPage: {r['page']} | Score: {r['score']:.3f}")
    print(r["text"][:700])
    print("=" * 70)


Page: 1 | Score: 0.871
Abstract— In this paper Arabic was investigated from the 
speech recognition problem point of view.  We propose a 
novel approach to build an Arabic Automated Speech 
Recognition System (ASR). This system is based on the open 
source CMU Sphinx-4, from the Carnegie Mellon University. 
CMU Sphinx is a large-vocabulary; speaker-independent, 
continuous speech recognition system based on discrete 
Hidden Markov Models (HMMs). We build a model using 
utilities from the OpenSource CMU Sphinx. We will 
demonstrate the possible adaptability of this system to Arabic 
voice recognition. 
 
Keywords: Speech recognition, Arabic language, HMMs, 
CMUSphinx-4, Artificial intelligence. 
 
 
I. INTRODUCTION

Page: 3 | Score: 0.862
nd there are as many syllables in a word as 
vowels in it [23]. 
B. Hello Arabic Digit application 
 
 
 
Table 1:  Ten Arabic digits, how to pronounce them, type of 
syllable, IPA representation [24] and number of syllables in 
every spoken digit [25

In [36]:
# Answer generation mein citation accuracy improve karo
def ask_uploaded_pdf(question, uploaded_chunks, uploaded_index, top_k=3):
    results = search_uploaded_pdf(
        question, uploaded_chunks, uploaded_index, top_k=top_k
    )

    if not results:
        return "Relevant information paper mein nahi mili."

    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    prompt = f"""You are ResearchLens AI, a research paper assistant.

Answer the question using only the context below.
Be concise and factual.
Cite the page number immediately after the claim, using [Page N].
Only cite pages that directly support the claim.
If the answer is not in the context, say so clearly.
Do not invent facts or page numbers.

Context:
{context}

Question: {question}

Answer:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 150
            }
        },
        timeout=180
    )

    response.raise_for_status()
    return response.json()["response"].strip()

In [37]:
print("Testing updated ResearchLens AI...")

test_chunks, test_index = load_pdf_cached(pdf_path)

answer = ask_uploaded_pdf(
    "Which speech recognition system is used in this paper?",
    test_chunks,
    test_index
)

print("\nAnswer:\n")
print(answer)

Testing updated ResearchLens AI...

Answer:

The speech recognition system used in this paper is the SPHINX system, which is based on Hidden Markov Models (HMMs). [Page 4]


In [38]:
# Retrieval ko answer ke liye verify karo
results = search_uploaded_pdf(
    "Which speech recognition system is used in this paper?",
    test_chunks,
    test_index,
    top_k=3
)

for r in results:
    print(f"\nPage: {r['page']} | Score: {r['score']:.3f}")
    print(r["text"][:350])
    print("-" * 50)


Page: 4 | Score: 0.641
, “An overview of the SPHINX 
speech recognition system,” IEEE Transactions on Acoustics, Speech 
and Signal Processing, vol. 38, no. 1, pp. 35–45, Jan. 1990. 
[6] 
X. Huang, F. Alleva, H. W. Hon, M. Y. Hwang, and R. Rosenfeld, “The 
SPHINX-II speech recognition system: an overview,” Computer Speech 
and Language, vol. 7, no. 2, pp. 137–148, 1993. 
--------------------------------------------------

Page: 1 | Score: 0.627
hat a person speaks 
into a microphone or telephone.  It has a wide area of 
applications: Command recognition (Voice user interface with 
the computer), Dictation, Interactive Voice Response, it can 
be used to learn a foreign language. ASR can help also, 
handicapped people to interact with society. It is a technology 
which makes life easier and
--------------------------------------------------

Page: 4 | Score: 0.582
ol. 2, 1993; K. F. Lee, 
“Automatic Speech Recognition the Development of the SPHINX 
System,” Kluwer Academic Publishers, 1

In [39]:
# Citation function update karo
def ask_uploaded_pdf(question, uploaded_chunks, uploaded_index, top_k=3):
    results = search_uploaded_pdf(
        question, uploaded_chunks, uploaded_index, top_k=top_k
    )

    if not results:
        return "Relevant information paper mein nahi mili."

    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    prompt = f"""You are ResearchLens AI.
Answer using only the provided context.
Be concise and factual.
Do not generate page citations yourself.
If the answer is not present in the context, say so.

Context:
{context}

Question: {question}

Answer:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0, "num_predict": 150}
        },
        timeout=180
    )
    response.raise_for_status()

    answer = response.json()["response"].strip()

    # Attach source pages from retrieved passages
    pages = sorted(set(r["page"] for r in results))
    sources = ", ".join(f"[Page {p}]" for p in pages)

    return f"{answer}\n\nRetrieved source pages: {sources}"

In [40]:
# last check
test_chunks, test_index = load_pdf_cached(pdf_path)

answer = ask_uploaded_pdf(
    "Which speech recognition system is used in this paper, and what models does it use?",
    test_chunks,
    test_index,
    top_k=3
)

print("ResearchLens AI — Final Test")
print("=" * 45)
print(answer)

ResearchLens AI — Final Test
The speech recognition system used in this paper is CMU Sphinx-4, which is based on Hidden Markov Models (HMMs). It uses three types of language-dependent models:

1. An acoustic model, which represents statistically a range of possible audio representations for the phonemes.
2. A pronunciation dictionary specifying how each word is pronounced in terms of the phonemes in the acoustic model.
3. A language model or grammar model.

Retrieved source pages: [Page 1], [Page 3], [Page 4]


In [41]:
# ui
upload_demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.
